# Approach 2: Text-to-SQL (LLM as a Query Generator)

Instead of feeding the LLM raw data, we teach it to **write SQL queries** that retrieve the answer.

**Model:** `gaussalgo/T5-LM-Large-text2sql-spider` — a T5-Large fine-tuned on the Spider Text-to-SQL benchmark.

**How it works:**
```
Input:  [DB schema description] + [Natural language question]
Output: SQL query
Then:   Execute SQL against actual database → return results
```

**Strengths:** Works on databases of any size, no data in the prompt, always up-to-date  
**Weaknesses:** Requires structured schema, may generate invalid SQL, needs execution layer

In [ ]:
# !pip install transformers torch pandas

In [ ]:
import pandas as pd
import sqlite3
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print(f"PyTorch version: {torch.__version__}")

## Step 1: Set up the database

In [ ]:
# Load CSV and push into SQLite
df = pd.read_csv('../data/employees.csv')

conn = sqlite3.connect(':memory:')  # in-memory DB
df.to_sql('employees', conn, if_exists='replace', index=False)

# Verify
result = pd.read_sql('SELECT * FROM employees LIMIT 5', conn)
print("Database loaded:")
result

In [ ]:
# Inspect schema (this is what we'll give the model)
schema_query = "SELECT sql FROM sqlite_master WHERE type='table' AND name='employees'"
schema = conn.execute(schema_query).fetchone()[0]
print("Table Schema:")
print(schema)

## Step 2: Load the Text-to-SQL model

**Model:** `gaussalgo/T5-LM-Large-text2sql-spider`  
- Architecture: T5-Large (783M parameters)
- Trained on: Spider benchmark (7,000+ complex SQL queries)
- Input format: `<schema> | <question>`

In [ ]:
MODEL_ID = "gaussalgo/T5-LM-Large-text2sql-spider"

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
model.eval()
print("Model loaded!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 3: Define input format and inference

In [ ]:
# The Spider-trained model expects this schema format
SCHEMA_DESCRIPTION = (
    "CREATE TABLE employees "
    "(id INTEGER, name TEXT, department TEXT, salary INTEGER, "
    "years_experience INTEGER, performance_score REAL, promoted TEXT)"
)

def text_to_sql(question: str, schema: str = SCHEMA_DESCRIPTION) -> str:
    """Convert a natural language question to SQL using the fine-tuned T5 model."""
    # The model was trained with this exact prompt format
    input_text = f"{schema} | {question}"
    
    inputs = tokenizer(
        input_text,
        return_tensors='pt',
        max_length=512,
        truncation=True
    )
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,       # beam search for better SQL
            early_stopping=True
        )
    
    sql = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return sql

def execute_sql(sql: str, connection) -> pd.DataFrame:
    """Execute SQL and return results as DataFrame."""
    try:
        return pd.read_sql(sql, connection)
    except Exception as e:
        return pd.DataFrame({'error': [str(e)]})

## Step 4: Ask questions — Input → SQL → Results

In [ ]:
questions = [
    "Who has the highest salary?",
    "How many employees work in Engineering?",
    "What is the average salary by department?",
    "List all employees who were promoted and have more than 5 years of experience.",
    "Which department has the highest average performance score?",
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    
    sql = text_to_sql(question)
    print(f"Generated SQL: {sql}")
    
    result = execute_sql(sql, conn)
    print(f"Result:")
    print(result.to_string(index=False))

## Step 5: Understanding what the model learned

In [ ]:
# The model maps natural language patterns to SQL constructs:
# "How many" → COUNT(*) or COUNT(col)
# "highest / most" → ORDER BY col DESC LIMIT 1 (or MAX)
# "average" → AVG(col)
# "by department" → GROUP BY department
# "who were promoted" → WHERE promoted = 'yes'

mapping_examples = [
    ("Count the number of promoted employees", "COUNT + WHERE"),
    ("Show the top 3 earners", "ORDER BY + LIMIT"),
    ("What is the minimum salary in Sales?", "MIN + WHERE"),
]

for question, expected_pattern in mapping_examples:
    sql = text_to_sql(question)
    print(f"Q: {question}")
    print(f"Expected pattern: {expected_pattern}")
    print(f"Generated SQL: {sql}")
    result = execute_sql(sql, conn)
    print(f"Result: {result.to_string(index=False)}")
    print()

## Key Takeaways

| | Text-to-SQL |
|---|---|
| **Setup** | Pre-trained model + any SQL database |
| **Scale** | Unlimited — queries the DB, not the model |
| **Accuracy** | ~80-85% on Spider benchmark; varies by complexity |
| **Cost** | Fixed per query (not per row) |
| **Best for** | Production databases, analytics, BI tools |

**Next:** See `03_tapas_table_qa.ipynb` where TaPas learns structural table relationships.